In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import pearsonr
from scipy.stats import bootstrap

fig_save_path = "/Users/thomassainsbury/Documents/Mathis_lab/Aug_Reg/AR_plots_new/"

In [2]:
cd /Users/thomassainsbury/Documents/Mathis_lab/Mathis_lab_code/FreelyMovingVR4Mice/dj_pipeline/

/Users/thomassainsbury/Documents/Mathis_lab/Mathis_lab_code/FreelyMovingVR4Mice/dj_pipeline


In [3]:
%run env_tom.py
%run run.py connect

2024-09-27 09:29:02,705::INFO::settings.py::Setting loglevel to INFO
2024-09-27 09:29:02,705::INFO::settings.py::Setting stores to {}
2024-09-27 09:29:02,706::INFO::settings.py::Setting database.misc.schema_prefix to 
2024-09-27 09:29:02,706::INFO::settings.py::Setting database.misc.create_tables to True
2024-09-27 09:29:02,707::INFO::settings.py::Setting enable_python_native_blobs to True
2024-09-27 09:29:02,707::INFO::settings.py::Setting database.host to 128.178.51.167:3309
2024-09-27 09:29:02,707::INFO::settings.py::Setting database.user to thomas
2024-09-27 09:29:02,708::INFO::settings.py::Setting database.password to thomas


Connecting thomas@128.178.51.167:3309


2024-09-27 09:29:03,003::INFO::connection.py::Connected thomas@128.178.51.167:3309
2024-09-27 09:29:03,108::INFO::table.py::could not log event in table ~log
2024-09-27 09:29:03,447::INFO::table.py::could not log event in table ~log


In [72]:
from vr4mice.schema import vr4mice, dlc, base_analysis
import vr4mice.analysis.plotting as plotting
import vr4mice.analysis.utils as utils
import vr4mice.analysis.visual_discrim_functions as vdf
from vr4mice.analysis import regression
from vr4mice.analysis.dlc_helpers import filter_dlc, getall_dlc_heading_angles, sync_dlc_w_game, _sync_dlc_w_game
vdf.get_rc_params()

In [80]:
def sync_keypoint_table(d, keypoint_cuttoff = 0.5, filter_window_length =10):
    # get time indexs from the game dataframe and the start time
    game_step_times = pd.DataFrame((base_analysis.DataFrame() & d).fetch("step_time", "step", "time_elapsed", "trial",as_dict=True)[0])
    start_time = (vr4mice.State() & d).fetch("start_time")[0]
    game_step_times ["start_time"]= start_time
    
    # Fetch the keypoint table and then sychronise with the game timesteps
    keypoint_df = (dlc.DLCKptsDf() & {"dataset": "Jacana_2024-07-29_2"}).get_all_data()[0]
    filt_dlc_df = filter_dlc(keypoint_df,cutoff=keypoint_cuttoff,window_length=filter_window_length)
    return(_sync_dlc_w_game(game_data=game_step_times,dlc=filt_dlc_df))
    

In [81]:
sync_key_point_table = sync_keypoint_table(d= {"dataset": "Jacana_2024-07-29_2"}) # add as table in dj

In [104]:
offline_dlc_variables = getall_dlc_heading_angles(sync_key_point_table.iloc[:,:-3]) # Then you can fetch that previous
offline_dlc_variables [["pose_time", "step_time", "step"]] = sync_key_point_table.iloc[:,-3:] # this readds the indexing columns save this in the synced_computed_variables_function

In [107]:

offline_dlc_variables.keys()

Index(['head_center_x', 'head_center_y', 'heading_dir', 'head_angle',
       'pose_time', 'step_time', 'step'],
      dtype='object')